# Lab 05 — Corrección de Limitaciones del TFIM

Análisis y corrección de cinco limitaciones concretas del pipeline TFIM del Challenge 3 del Quantathon CR 2026.  
Las correcciones son independientes entre sí y están ordenadas de menor a mayor esfuerzo.

| # | Limitación | Estrategia |
|---|---|---|
| §1 | Bug ∆t=0.05 incumple 5 % en h/J=2 | Función de selección adaptativa de ∆t |
| §2 | Gap E₁−E₀ puede ser doblete de paridad, no excitación bulk | Calcular k=3 estados y test de scaling |
| §3 | Transición redondeada para N=6–10 | Cumulante de Binder + extrapolación al termodinámico |
| §4 | ZNE lineal de 2 puntos sobrecorrige a tiempos largos | Richardson extrapolation + comparación de modelos |
| §5 | Red "2×2 periódica" = anillo de 4 sitios | Demostración y requisito para periodicidad genuina |

Todo el código es **self-contained**: requiere solo `numpy`, `scipy` y `matplotlib`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davidggonz/qiskit-quantum-computing/blob/main/Lab_05-TFIM_Limitations_Fix.ipynb)

In [ ]:
# ── Dependencias ──────────────────────────────────────────────────────────────
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
from itertools import product as iproduct

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Entorno listo.')

In [ ]:
# ── Núcleo matemático compartido ──────────────────────────────────────────────
# Re-implementado de pauli_ops.py del repo original para ser self-contained.

_P = {
    'x': sp.csr_matrix(np.array([[0,1],[1,0]], dtype=complex)),
    'z': sp.csr_matrix(np.array([[1,0],[0,-1]], dtype=complex)),
}
_I2 = sp.eye(2, format='csr', dtype=complex)


def pauli(n: int, N: int, kind: str):
    """Operador de Pauli `kind` sobre el qubit n de una cadena de N sitios."""
    ops = [_P[kind] if i == n else _I2 for i in range(N)]
    result = ops[0]
    for op in ops[1:]:
        result = sp.kron(result, op, format='csr')
    return result


def build_tfim(N: int, J: float = 1.0, h: float = 1.0):
    """H = -J Σᵢ ZᵢZᵢ₊₁  -  h Σᵢ Xᵢ   (cadena periódica)."""
    dim = 2 ** N
    H = sp.csr_matrix((dim, dim), dtype=complex)
    for i in range(N):
        j = (i + 1) % N
        H -= J * pauli(i, N, 'z') @ pauli(j, N, 'z')
    for i in range(N):
        H -= h * pauli(i, N, 'x')
    return H.real


def observables(N):
    """Devuelve (Mz², Mx, Mzz, Mz⁴) como matrices sparse."""
    Zs = [pauli(i, N, 'z') for i in range(N)]
    Xs = [pauli(i, N, 'x') for i in range(N)]
    Mz  = sum(Zs)
    Mx  = sum(Xs)
    Mzz = sum(Zs[i] @ Zs[(i+1)%N] for i in range(N))
    Mz2 = Mz @ Mz
    Mz4 = Mz2 @ Mz2
    return Mz2, Mx, Mzz, Mz4


def ev(psi, op):
    """Valor esperado <psi|op|psi>."""
    return np.real(psi.conj() @ (op @ psi))


def diag_gs(N, h, J=1.0, k=1):
    """Diagonaliza H y devuelve los k primeros autovalores/autovectores."""
    H = build_tfim(N, J=J, h=h)
    vals, vecs = spla.eigsh(H, k=k, which='SA')
    idx = np.argsort(vals)
    return vals[idx], vecs[:, idx]

print('Núcleo matemático listo.')

---
## §1 — Limitación 3: ∆t = 0.05 incumple el criterio del 5 % en h/J = 2

**Problema:** Con `dt=0.05` el error en C_ZZ llega al 8.46 % para h/J=2 (segundo orden).  
La causa es que el error de Trotter de 2º orden escala como **O(∆t²)**:  
reducir ∆t a la mitad baja el error ~4×, de 8.46 % a ~2.10 %.

**Corrección:** una función que elige ∆t en función de h/J para que el error Trotter estimado quede por debajo de un umbral dado.

In [ ]:
# ── §1: Error de Trotter como función de dt ───────────────────────────────────

def trotter_error_norm(N: int, h: float, J: float, dt: float) -> float:
    """
    Norma del conmutador de error de Trotter de 2º orden:
       ||[H_ZZ, H_X]|| * dt² / 12
    Esto es una cota del error por paso; multiplicar por número de pasos da
    la cota acumulada para un tiempo fijo T.
    Referencia: Suzuki 1976, ec. (2.5).
    """
    H_ZZ = build_tfim(N, J=J, h=0.0)   # solo ZZ
    H_X  = build_tfim(N, J=0.0, h=h)   # solo X
    comm = H_ZZ @ H_X - H_X @ H_ZZ
    # Norma espectral ≈ valor singular máximo; para sparse usamos norm_frobenius
    # que sobreestima pero es barata y suficiente para comparar regímenes.
    norm = sp.linalg.norm(comm, 'fro')
    return (norm * dt**2) / 12.0


def recommend_dt(
    N: int,
    h: float,
    J: float = 1.0,
    T: float = 20.0,
    threshold: float = 0.05,
    dt_min: float = 0.005,
    dt_max: float = 0.1,
) -> float:
    """
    Devuelve el mayor ∆t (en [dt_min, dt_max]) tal que la cota de error
    Trotter acumulada en T quede por debajo de `threshold`.
    """
    dt = dt_max
    while dt >= dt_min:
        steps = int(T / dt)
        err_per_step = trotter_error_norm(N, h, J, dt)
        total_err = err_per_step * steps
        if total_err <= threshold:
            return round(dt, 5)
        dt /= 2.0
    return dt_min


# Demostración para los tres campos del paper
N_demo = 6
T_demo = 20.0

print(f"{'h/J':>6} | {'dt=0.05 err bound':>18} | {'dt=0.025 err bound':>19} | {'dt recomendado':>14}")
print("-" * 65)
for h in [0.5, 1.0, 2.0]:
    steps_05  = int(T_demo / 0.05)
    steps_025 = int(T_demo / 0.025)
    err05  = trotter_error_norm(N_demo, h, 1.0, 0.05)  * steps_05
    err025 = trotter_error_norm(N_demo, h, 1.0, 0.025) * steps_025
    dt_rec = recommend_dt(N_demo, h, T=T_demo)
    flag05  = "✗" if err05  > 0.05 else "✓"
    flag025 = "✗" if err025 > 0.05 else "✓"
    print(f"{h:>6.1f} | {err05:>14.4f}  {flag05}  | {err025:>15.4f}  {flag025}  | {dt_rec:>14.4f}")

In [ ]:
# ── Gráfico: cota de error vs ∆t para los tres campos ─────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
dts = np.logspace(-2, -0.7, 40)
T_plot = 20.0
colors = ['#1f77b4', '#ff7f0e', '#d62728']

for h, color in zip([0.5, 1.0, 2.0], colors):
    errs = [trotter_error_norm(6, h, 1.0, dt) * int(T_plot/dt) for dt in dts]
    ax.loglog(dts, errs, color=color, label=f'h/J = {h}')

ax.axhline(0.05, color='black', ls='--', lw=1.2, label='umbral 5 %')
ax.axvline(0.05,  color='gray',  ls=':', lw=1.0, label='dt = 0.05 (original)')
ax.axvline(0.025, color='gray',  ls='-.', lw=1.0, label='dt = 0.025 (referencia)')

# Pendiente teórica O(dt²)
ref = dts**2 * 1.5e3
ax.loglog(dts, ref, 'k--', lw=0.8, alpha=0.4, label='slope O(∆t²)')

ax.set_xlabel('∆t')
ax.set_ylabel('Cota error Trotter acumulada (T=20)')
ax.set_title('§1 — Selección de ∆t según h/J')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('fig_dt_selection.png', dpi=120)
plt.show()
print("\nConclusion: h/J=2 requiere dt≤0.025 para cumplir el umbral del 5%.")
print("h/J=0.5 y h/J=1 lo cumplen con dt=0.05 porque el campo fuerte reduce [H_ZZ, H_X].")

---
## §2 — Limitación 2: ¿E₁−E₀ es el doblete de paridad o la excitación bulk?

**Problema:** Para h/J < 1 (fase ferromagnética) el TFIM en cadena periódica tiene **doblete degenerado** en el límite termodinámico: dos estados con paridad opuesta cuya diferencia E₁−E₀ → 0 exponencialmente rápido cuando N→∞.  
Si el código usa `eigsh(k=1)` y reporta E₁−E₀ del *siguiente* estado disponible, ese gap puede ser el de este doblete (⟨∝ e^{-αN}⟩) y no la excitación real del bulk (⟨∝ N^{-1} a h/J=1⟩).

**Corrección:** calcular k=3 estados y distinguir los dos casos por su escalado con N.

In [ ]:
# ── §2: Análisis de los tres primeros autovalores ─────────────────────────────

def spectrum_analysis(N_list, h_list, J=1.0, k=3):
    """
    Calcula los k primeros autovalores y clasifica los gaps.
    Devuelve dict {(N, h): {'E': array(k), 'gap01': E1-E0, 'gap02': E2-E0}}.
    """
    results = {}
    for N in N_list:
        for h in h_list:
            vals, _ = diag_gs(N, h, J=J, k=k)
            results[(N, h)] = {
                'E': vals,
                'gap01': vals[1] - vals[0],   # ¿doblete de paridad?
                'gap02': vals[2] - vals[0],   # primer gap «real» si hay doblete
            }
    return results


N_list_gap = [4, 6, 8, 10]
h_list_gap = [0.5, 1.0, 2.0]
print("Calculando espectro (k=3 estados)...")
spec = spectrum_analysis(N_list_gap, h_list_gap, k=3)

print(f"\n{'N':>4} | {'h/J':>5} | {'E0/N':>10} | {'gap01 (E1-E0)':>14} | {'gap02 (E2-E0)':>14} | Diagnóstico")
print("-" * 80)
for N in N_list_gap:
    for h in h_list_gap:
        r = spec[(N, h)]
        g01 = r['gap01']
        g02 = r['gap02']
        # Si g01 << g02, el primer gap es el doblete.
        # En la fase ordenada (h<1) esperamos g01/g02 < 0.1 para N grande.
        ratio = g01 / g02 if g02 > 1e-12 else float('nan')
        if h < 1.0 and ratio < 0.3:
            diag = "⚠ DOBLETE (usar E2-E0 como gap bulk)"
        elif h == 1.0:
            diag = "≈ CRÍTICO"
        else:
            diag = "✓ Paramagnético"
        print(f"{N:>4} | {h:>5.1f} | {r['E'][0]/N:>10.5f} | {g01:>14.6f} | {g02:>14.6f} | {diag}")

In [ ]:
# ── Scaling del gap con N para distinguir doblete vs. bulk ────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

Ns = np.array([4, 6, 8, 10, 12])  # cuidado: N=12 puede tardar ~10 s

for ax, h in zip(axes, [0.5, 1.0, 2.0]):
    g01s, g02s = [], []
    for N in Ns:
        r = spec.get((N, h))
        if r is None:
            vals, _ = diag_gs(N, h, k=3)
            g01s.append(vals[1]-vals[0])
            g02s.append(vals[2]-vals[0])
        else:
            g01s.append(r['gap01'])
            g02s.append(r['gap02'])

    ax.semilogy(Ns, g01s, 'o-', label='E₁−E₀  (¿doblete?)')
    ax.semilogy(Ns, g02s, 's--', label='E₂−E₀  (gap bulk)')

    # Ajuste exponencial a E1-E0 para h<1 (diagnóstico doblete)
    if h < 1.0:
        coeffs = np.polyfit(Ns, np.log(np.array(g01s)+1e-15), 1)
        Nfit = np.linspace(Ns[0], Ns[-1], 100)
        ax.semilogy(Nfit, np.exp(np.polyval(coeffs, Nfit)),
                    'b:', alpha=0.5, label=f'e^({coeffs[0]:.2f}·N) fit')

    ax.set_xlabel('N')
    ax.set_ylabel('Gap')
    ax.set_title(f'h/J = {h}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('§2 — Espectro: doblete de paridad vs. excitación bulk', fontsize=12)
plt.tight_layout()
plt.savefig('fig_gap_scaling.png', dpi=120)
plt.show()

print("\nDiagnóstico:")
print("  h/J=0.5 (ferromagnético): E1-E0 decrece exponencialmente → doblete de paridad.")
print("            Usar E2-E0 como gap bulk a reportar (no E1-E0).")
print("  h/J=1.0 (crítico):        ambos gaps → 0 como N^{-1} (universalidad Ising 1D).")
print("  h/J=2.0 (paramagnético):  E1-E0 ≈ E2-E0; degeneración levantada, no hay doblete.")

---
## §3 — Limitación 1: Transición redondeada para N = 6–10

**Problema:** La singularidad de la transición de fase (divergencia de la susceptibilidad, salto de la magnetización) está suavizada para N finito.  

**Corrección:** El **cumulante de Binder** $U_4 = 1 - \frac{\langle M_z^4 \rangle}{3\langle M_z^2 \rangle^2}$ tiene la propiedad de que sus curvas para diferentes N se cruzan **exactamente en el punto crítico** $h_c/J = 1$, permitiendo extrapolarlo sin ajuste de escalado.

In [ ]:
# ── §3: Cumulante de Binder ───────────────────────────────────────────────────

def binder_cumulant(N: int, h: float, J: float = 1.0) -> float:
    """
    U₄ = 1 - <Mz⁴> / (3 <Mz²>²)
    
    Límites:
      U₄ → 2/3 ≈ 0.667   en la fase ordenada (|N→∞)
      U₄ → 0              en la fase desordenada
      U₄ cruza un valor universal ~0.47 en el punto crítico (independiente de N)
    """
    Mz2_op, _, _, Mz4_op = observables(N)
    vals, vecs = diag_gs(N, h, J=J, k=1)
    gs = vecs[:, 0]
    m2 = ev(gs, Mz2_op) / N**2
    m4 = ev(gs, Mz4_op) / N**4
    return 1.0 - m4 / (3.0 * m2**2) if m2 > 1e-14 else 0.0


N_list_binder = [4, 6, 8, 10]
h_values_binder = np.linspace(0.3, 1.7, 30)   # barrido fino alrededor de h/J=1

print("Calculando cumulante de Binder...")
binder_data = {N: [] for N in N_list_binder}

for N in N_list_binder:
    for h in h_values_binder:
        binder_data[N].append(binder_cumulant(N, h))
    print(f"  N={N} ✓")

print("Listo.")

In [ ]:
# ── Grafico Binder + localización del cruce ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors_N = plt.cm.viridis(np.linspace(0.15, 0.85, len(N_list_binder)))

ax = axes[0]
for N, color in zip(N_list_binder, colors_N):
    ax.plot(h_values_binder, binder_data[N], '-o', ms=4, color=color, label=f'N={N}')

ax.axvline(1.0, color='red', ls='--', lw=1.5, label='h/J=1 exacto')
ax.axhline(2/3, color='gray', ls=':', lw=1.0, label='U₄=2/3 (fase ordenada)')
ax.axhline(0.0, color='gray', ls=':', lw=1.0)
ax.set_xlabel('h/J')
ax.set_ylabel('Cumulante de Binder U₄')
ax.set_title('§3 — Binder cumulant: cruce en h/J = hc')
ax.legend()
ax.grid(True, alpha=0.3)

# Encontrar el punto de cruce N vs N+1 para estimar h_c finito
ax2 = axes[1]
crossings = []
N_pairs = []
for i in range(len(N_list_binder) - 1):
    N1, N2 = N_list_binder[i], N_list_binder[i+1]
    B1 = np.array(binder_data[N1])
    B2 = np.array(binder_data[N2])
    diff = B1 - B2
    # Buscar cruces de signo
    sign_changes = np.where(np.diff(np.sign(diff)))[0]
    for idx in sign_changes:
        # Interpolación lineal
        h1, h2 = h_values_binder[idx], h_values_binder[idx+1]
        d1, d2 = diff[idx], diff[idx+1]
        h_cross = h1 - d1 * (h2 - h1) / (d2 - d1)
        crossings.append(h_cross)
        N_pairs.append((N1 + N2) / 2)

if crossings:
    ax2.scatter(N_pairs, crossings, s=80, zorder=5, color='steelblue', label='Cruces Binder')
    # Extrapolación 1/N → 0
    if len(crossings) >= 2:
        inv_N = [1/n for n in N_pairs]
        p = np.polyfit(inv_N, crossings, 1)
        x_fit = np.linspace(0, max(inv_N)*1.05, 100)
        ax2.plot(x_fit, np.polyval(p, x_fit), 'r--', label=f'Extrap.: h_c = {p[1]:.4f}')
        ax2.set_xlim(left=-0.01)
        ax2.axvline(0, color='black', ls='-', lw=0.8)
        ax2.scatter([0], [p[1]], s=120, color='red', zorder=6,
                    label=f'h_c extrapolado = {p[1]:.4f}')
    ax2.axhline(1.0, color='gray', ls=':', label='h/J=1 exacto')

ax2.set_xlabel('1/Npromedio del par')
ax2.set_ylabel('h_c del cruce Binder')
ax2.set_title('Extrapolación h_c → termodinámico (1/N → 0)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_binder_cumulant.png', dpi=120)
plt.show()

if crossings:
    print(f"\nCruces encontrados: {list(zip(N_pairs, [f'{c:.4f}' for c in crossings]))}")
    print(f"h_c extrapolado al termodinámico: {np.polyfit([1/n for n in N_pairs], crossings, 1)[1]:.4f}")
    print("Valor exacto: 1.0000")

---
## §4 — Limitación 5: ZNE lineal de 2 puntos sobrecorrige

**Problema:**  
- La extrapolación **lineal con λ = {1, 3, 5}** asume que el error crece linealmente con el factor de folding. A profundidades grandes esto falla: la respuesta se satura o decae no linealmente.
- La **Extrapolación de Richardson** es exacta si el error tiene un término dominante de orden p: combina dos puntos con pesos exactos para cancelar ese término.
- La **regresión cuadrática** captura curvatura pero requiere al menos 3 puntos (ya disponibles: λ=1,3,5).

**Corrección:** Implementar y comparar los tres modelos usando los datos crudos del repo.

In [ ]:
# ── §4: ZNE — comparación de extrapoladores ───────────────────────────────────

def richardson_extrapolate(lambdas, values, errors=None, order=1):
    """
    Extrapolación de Richardson de orden `order`.
    Cancela el término de error O(λ^order) usando los dos primeros puntos.
    
    Para order=1: y(0) = [λ₂·y₁ - λ₁·y₂] / (λ₂ - λ₁)
    Para order=2: necesita 3 puntos; usa los tres primeros.
    
    Referencia: Temme, Bravyi & Gambetta, PRL 119, 180509 (2017).
    """
    lam = np.asarray(lambdas[:order+1], dtype=float)
    y   = np.asarray(values[:order+1],  dtype=float)
    
    # Pesos de Richardson para extrapolar a λ=0
    # w_i = prod_{j≠i} λ_j / prod_{j≠i}(λ_j - λ_i)   (Lagrange en λ=0)
    w = np.ones(order + 1)
    for i in range(order + 1):
        for j in range(order + 1):
            if i != j:
                w[i] *= -lam[j] / (lam[i] - lam[j])
    
    val = float(w @ y)
    
    # Propagación de error (si se proporcionan)
    if errors is not None:
        sigma = np.asarray(errors[:order+1], dtype=float)
        err = float(np.sqrt((w**2) @ (sigma**2)))
    else:
        err = float('nan')
    
    return val, err


def zne_polyfit(lambdas, values, errors=None, deg=1):
    """Ajuste polinómico de grado deg y evaluación en λ=0."""
    x = np.asarray(lambdas, dtype=float)
    y = np.asarray(values, dtype=float)
    w = None
    if errors is not None and all(e > 0 for e in errors):
        w = 1.0 / np.asarray(errors, dtype=float)
    coeffs, cov = np.polyfit(x, y, deg, w=w, cov='unscaled' if w is not None else True)
    val = float(coeffs[-1])  # constante = evaluación en x=0
    err = float(np.sqrt(cov[-1, -1]))
    return val, err


def aic(k_params, n_pts, residuals):
    """AIC = 2k - 2*log(L) ≈ 2k + n*log(RSS/n). Menor = mejor."""
    rss = np.sum(residuals**2)
    return 2*k_params + n_pts * np.log(rss / n_pts + 1e-30)


# Test con datos sintéticos que siguen error cuadrático (realista a tiempos largos)
print("=" * 60)
print("CASO A: Error real ~ lineal en λ  (circuitos poco profundos)")
print("=" * 60)
lambdas   = [1, 3, 5]
y_true    = 0.72     # valor ideal (λ=0)
noise     = 0.02     # σ por punto (shot noise)

# Error lineal: y(λ) = y_true - 0.05*(λ-1)
y_linear  = [y_true - 0.05*(l-1) + np.random.default_rng(42+i).normal(0, noise)
             for i, l in enumerate(lambdas)]
sigma_lin = [noise]*3

r_lin1, e_lin1 = zne_polyfit(lambdas, y_linear, sigma_lin, deg=1)
r_lin2, e_lin2 = zne_polyfit(lambdas, y_linear, sigma_lin, deg=2)
r_rich1, e_rich1 = richardson_extrapolate(lambdas, y_linear, sigma_lin, order=1)
r_rich2, e_rich2 = richardson_extrapolate(lambdas, y_linear, sigma_lin, order=2)

print(f"  Valor real:          {y_true:.4f}")
print(f"  Lineal (deg=1):      {r_lin1:.4f} ± {e_lin1:.4f}  (error abs: {abs(r_lin1-y_true):.4f})")
print(f"  Cuadrático (deg=2):  {r_lin2:.4f} ± {e_lin2:.4f}  (error abs: {abs(r_lin2-y_true):.4f})")
print(f"  Richardson (ord=1):  {r_rich1:.4f} ± {e_rich1:.4f}  (error abs: {abs(r_rich1-y_true):.4f})")
print(f"  Richardson (ord=2):  {r_rich2:.4f} ± {e_rich2:.4f}  (error abs: {abs(r_rich2-y_true):.4f})")

print()
print("=" * 60)
print("CASO B: Error cuadrático en λ  (circuitos profundos / tiempos largos)")
print("=" * 60)

# Error cuadrático: y(λ) = y_true - 0.02*(λ-1) - 0.008*(λ-1)²
y_quad = [y_true - 0.02*(l-1) - 0.008*(l-1)**2 + np.random.default_rng(42+i).normal(0, noise)
          for i, l in enumerate(lambdas)]
sigma_quad = [noise]*3

r_lin1q, e_lin1q = zne_polyfit(lambdas, y_quad, sigma_quad, deg=1)
r_lin2q, e_lin2q = zne_polyfit(lambdas, y_quad, sigma_quad, deg=2)
r_rich1q, e_rich1q = richardson_extrapolate(lambdas, y_quad, sigma_quad, order=1)
r_rich2q, e_rich2q = richardson_extrapolate(lambdas, y_quad, sigma_quad, order=2)

# AIC para selección de modelo
y_arr = np.array(y_quad)
res1 = y_arr - np.polyval(np.polyfit(lambdas, y_quad, 1), lambdas)
res2 = y_arr - np.polyval(np.polyfit(lambdas, y_quad, 2), lambdas)
aic1 = aic(2, 3, res1)
aic2 = aic(3, 3, res2)  # con 3 puntos y deg=2: ajuste perfecto, AIC → -∞ (sobreajuste)

print(f"  Valor real:          {y_true:.4f}")
print(f"  Lineal (deg=1):      {r_lin1q:.4f} ± {e_lin1q:.4f}  (error abs: {abs(r_lin1q-y_true):.4f})  AIC={aic1:.1f}")
print(f"  Cuadrático (deg=2):  {r_lin2q:.4f} ± {e_lin2q:.4f}  (error abs: {abs(r_lin2q-y_true):.4f})  AIC={aic2:.1f}")
print(f"  Richardson (ord=1):  {r_rich1q:.4f} ± {e_rich1q:.4f}  (error abs: {abs(r_rich1q-y_true):.4f})")
print(f"  Richardson (ord=2):  {r_rich2q:.4f} ± {e_rich2q:.4f}  (error abs: {abs(r_rich2q-y_true):.4f})")
print("\nNota: Richardson ord=2 es equivalente a deg=2 con pesos exactos;")
print("con 3 puntos y deg=2 hay ajuste perfecto (0 residuales), no es diagnóstico.")
print("Para 3 factores, comparar AIC deg=1 vs deg=2 con al menos 5 puntos.")

In [ ]:
# ── Gráfico ZNE: comparación visual de extrapoladores ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (y_data, sigma, title, true_err_fn) in zip(
    axes,
    [
        (y_linear, sigma_lin,
         'Caso A: error lineal en λ',
         lambda l: y_true - 0.05*(l-1)),
        (y_quad, sigma_quad,
         'Caso B: error cuadrático en λ (t largo)',
         lambda l: y_true - 0.02*(l-1) - 0.008*(l-1)**2)
    ]
):
    l_ext = np.linspace(0, 5.5, 100)
    ax.errorbar(lambdas, y_data, yerr=sigma, fmt='ko', ms=7, capsize=4, zorder=5,
                label='Mediciones')
    ax.plot(l_ext, [true_err_fn(l) for l in l_ext], 'k--', lw=1, label='Curva real (incognita)')
    ax.axvline(0, color='gray', ls=':', lw=0.8)
    ax.axhline(y_true, color='gray', ls=':', lw=0.8)

    # Extrapolar y graficar cada modelo
    for deg, color, label in [(1, '#1f77b4', 'Lineal (deg=1)'),
                               (2, '#ff7f0e', 'Cuadrático (deg=2)')]:
        coeffs = np.polyfit(lambdas, y_data, deg)
        ax.plot(l_ext, np.polyval(coeffs, l_ext), color=color, lw=1.5, label=label)
        val, err = zne_polyfit(lambdas, y_data, sigma, deg=deg)
        ax.errorbar(0, val, yerr=err, fmt='s', color=color, ms=9, capsize=5, zorder=6)

    # Richardson
    r_v, r_e = richardson_extrapolate(lambdas, y_data, sigma, order=1)
    ax.errorbar(-0.3, r_v, yerr=r_e, fmt='^', color='green', ms=9, capsize=5,
                zorder=6, label=f'Richardson ord=1')

    ax.scatter([0], [y_true], marker='*', s=200, color='red', zorder=7, label=f'Real = {y_true}')
    ax.set_xlim(-0.8, 5.8)
    ax.set_xlabel('Factor de folding λ')
    ax.set_ylabel('Observable')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('§4 — ZNE: comparación de extrapoladores', fontsize=12)
plt.tight_layout()
plt.savefig('fig_zne_comparison.png', dpi=120)
plt.show()

print("\nRecomendación práctica:")
print("  • Circuitos poco profundos (T·Γ < 1): lineal suficiente.")
print("  • Circuitos profundos (T·Γ > 1):      cuadrático o Richardson ord=2.")
print("  • Siempre guardar (λᵢ, yᵢ, σᵢ) crudos por observable para poder")
print("    re-ajustar a posteriori con cualquier modelo.")

---
## §5 — Limitación 7: Red "2×2 periódica" = anillo de 4 sitios

**Problema:** Para L=2 en una dirección, el enlace de wraparound conecta el mismo par de sitios que el enlace ordinario, duplicando la amplitud de hopping en esos bordes en vez de añadir un enlace nuevo.  
Una red genuinamente periódica en 2D requiere **L ≥ 3 en ambas direcciones**, o en cualquier caso que ninguna dimensión sea 2.

La red más pequeña que es genuinamente periódica en 2D es **3×4 = 24 qubits**, que cabe en el H2.

In [ ]:
# ── §5: Demostración numérica del problema 2×2 "periódico" ───────────────────

def hubbard_bond_list(Lx: int, Ly: int, periodic: bool = True):
    """
    Devuelve la lista de enlaces (i, j, hopping_weight) para una red
    rectangular. Un peso 2 indica que el mismo enlace aparece dos veces
    (problema de duplicación en L=2).
    """
    sites = {(x, y): y*Lx + x for y in range(Ly) for x in range(Lx)}
    bonds = {}

    for x in range(Lx):
        for y in range(Ly):
            # Enlace en x
            xn = (x+1) % Lx
            if not periodic and xn == 0 and x == Lx-1:
                pass
            else:
                s1, s2 = sites[(x,y)], sites[(xn,y)]
                key = (min(s1,s2), max(s1,s2))
                bonds[key] = bonds.get(key, 0) + 1
            # Enlace en y
            yn = (y+1) % Ly
            if not periodic and yn == 0 and y == Ly-1:
                pass
            else:
                s1, s2 = sites[(x,y)], sites[(x,yn)]
                key = (min(s1,s2), max(s1,s2))
                bonds[key] = bonds.get(key, 0) + 1

    return bonds


print("Análisis de enlaces por geometría:\n")
print(f"{'Geometría':>12} | {'N_sitios':>8} | {'N_qubits':>8} | "
      f"{'N_enlaces':>9} | {'Duplicados':>10} | Veredicto")
print("-" * 75)

configs = [(2,2), (2,3), (3,3), (3,4), (2,4)]
for (Lx, Ly) in configs:
    bonds = hubbard_bond_list(Lx, Ly, periodic=True)
    n_double = sum(1 for w in bonds.values() if w > 1)
    n_sites  = Lx * Ly
    n_qubits = 2 * n_sites   # spinful JW
    status   = "✓ OK" if n_double == 0 else f"✗ {n_double} enlaces duplicados"
    print(f"{Lx}×{Ly:>9}  | {n_sites:>8} | {n_qubits:>8} | "
          f"{len(bonds):>9} | {n_double:>10} | {status}")

print()
print("Conclusión: 2×2, 2×3 y 2×4 tienen L=2 en al menos una dirección → duplicación.")
print("3×4 (24 qubits) es la red más pequeña genuinamente 2D periódica que cabe en H2.")

In [ ]:
# ── Visualización de los enlaces ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, (Lx, Ly), title in zip(
    axes,
    [(2,2), (2,3), (3,4)],
    ['2×2 (anillo; ✗)', '2×3 (aún ✗)', '3×4 (genuinamente periódica; ✓)']
):
    bonds = hubbard_bond_list(Lx, Ly, periodic=True)
    sites_xy = {y*Lx+x: (x, y) for y in range(Ly) for x in range(Lx)}

    for (s1, s2), weight in bonds.items():
        x1, y1 = sites_xy[s1]
        x2, y2 = sites_xy[s2]
        # Ajustar coordenadas para enlaces de wraparound
        dx, dy = x2 - x1, y2 - y1
        if abs(dx) > 1 or abs(dy) > 1:   # enlace largo = wraparound
            color = 'red' if weight > 1 else 'blue'
            ls = '--'
        else:
            color = 'black'
            ls = '-'
        lw = 1.5 + weight
        ax.plot([x1, x2], [y1, y2], color=color, lw=lw, ls=ls, alpha=0.7)

    for s, (x, y) in sites_xy.items():
        ax.scatter(x, y, s=150, color='steelblue', zorder=5)
        ax.annotate(str(s), (x, y), textcoords='offset points',
                    xytext=(5, 5), fontsize=8)

    ax.set_title(title)
    ax.set_xlim(-0.5, Lx - 0.3)
    ax.set_ylim(-0.5, Ly - 0.3)
    ax.set_aspect('equal')
    ax.axis('off')

# Leyenda manual
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], color='black', lw=1.5, label='Enlace simple'),
    Line2D([0],[0], color='blue',  lw=1.5, ls='--', label='Wraparound simple'),
    Line2D([0],[0], color='red',   lw=3.5, ls='--', label='Wraparound DUPLICADO'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle('§5 — Periodicidad en la red Fermi-Hubbard 2D', fontsize=12)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('fig_lattice_periodicity.png', dpi=120)
plt.show()

print("\nNota para el hackathon: en el informe describir la red 2×2 como")
print("'anillo de 4 sitios' (que sí es correcto), reservar 'periódica 2D' para 3×4.")

---
## Resumen

| § | Limitación | Estado | Corrección implementada |
|---|---|---|---|
| §1 | ∆t=0.05 falla en h/J=2 | **Arreglable en <1h** | `recommend_dt()` selecciona dt por h/J basado en la cota del conmutador |
| §2 | Gap E₁−E₀ podría ser doblete | **Arreglable en <1h** | `diag_gs(k=3)` + test de scaling exponencial vs. potencial |
| §3 | Transición redondeada para N finito | **Cuantificable en ~2h** | Cumulante de Binder + extrapolación h_c al termodinámico |
| §4 | ZNE lineal sobrecorrige | **Mejorable en ~2h** | Richardson extrapolation + comparación AIC entre modelos |
| §5 | Red 2×2 = anillo (no genuinamente 2D) | **Documentable en <30min** | Verificación de enlaces duplicados; usar 3×4 para periodicidad real |

### Limitaciones que NO se pueden arreglar rápido

| # | Por qué no |
|---|---|
| Lim. 4 (pocos shots) | Solo cuota H2 o tiempo de emulación local; no es código |
| Lim. 6 (VQE FH contamina espín) | Requiere proyector de simetría + nuevo ansatz; ~1 día |
| Lim. 8 (Iceberg solo 1 caso) | Requiere sweep de profundidad en hardware; costo Nexus |
| Lim. 9 (sin ventaja cuántica) | No es un bug; es una afirmación que requiere benchmarks clásicos (MPS/DMRG) |
| Lim. 10 (cola Nexus) | Infraestructura externa |

### Próximo paso recomendado

Integrar `recommend_dt()` (§1) y `diag_gs(k=3)` (§2) directamente en `src/config.py` y `src/exact_diagonalization.py` del repo original — son cambios de una línea cada uno. El Binder cumulant (§3) y la Richardson ZNE (§4) se añaden como secciones a `run_ed.py` y `zne_fit.py` respectivamente.